# Tarea 4 — Pipeline completo (Colab)

De los audios a registros estructurados y verificados: benchmark del ASR (GPU + CPU), extracción con dos caminos
(reglas y LLM), validación y guardado de todos los resultados.

**Antes de empezar**
1. Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU** (T4).
2. **No hace falta ninguna clave.** Por defecto (`pipeline.extractor: rules` en `config.yaml`) la extracción usa solo reglas, sin API.
   Solo si cambias esa opción para usar el extractor LLM (de pago): panel izquierdo 🔑 **Secrets** → `ANTHROPIC_API_KEY` y acceso al notebook.
3. Los 10 audios ya vienen en el repo (`tasks/*/audio/`). Puedes subir otros en el paso 4.

**Tiempo:** la GPU tarda unos 7 minutos con todo; la CPU es muy lenta (en Colab, `medium` ~1,5 min por audio y `large-v3` bastante más: usa `MODELOS_CPU` para limitarla). Las transcripciones se guardan en
`transcriptions_cache/`, así que si algo se corta puedes volver a ejecutar y continúa donde quedó.

## 1. Obtener el código

In [ ]:
# Si estás en Colab el repo no está: se clona. Si ya estás dentro de la carpeta de la Tarea 4 (p. ej. en local) no se hace nada.
import os
from pathlib import Path

REPO = "https://github.com/RenataZuta/Homework.git"
RAMA = "main"  # cambia a tu rama (p. ej. "tarea4") si el engine aún no está en main

if not Path("engine/core.py").exists():
    !git clone --depth 1 --branch {RAMA} {REPO} /content/Homework
    %cd "/content/Homework/Tarea Septiembre/Tarea 4"
print("Carpeta de trabajo:", os.getcwd())

## 2. Instalar requirements

In [ ]:
# requirements.txt fija numpy, pandas, scipy y streamlit, pero Colab ya trae sus propias versiones de numpy/pandas/scipy
# (cambiarlas obliga a reiniciar el entorno) y streamlit es solo para la interfaz: se instala todo lo demás.
!grep -v -i -E "^(numpy|pandas|scipy|streamlit)" requirements.txt > /tmp/req_colab.txt
# CTranslate2 necesita las librerías CUDA 12 y el torch de Colab trae CUDA 13 (este paquete solo existe en Linux).
!echo 'nvidia-cublas-cu12; sys_platform == "linux"' >> /tmp/req_colab.txt
!pip install -q -r /tmp/req_colab.txt

## 3. Imports, GPU y configuración

In [ ]:
import ctypes
import importlib.util
import json
import shutil
import sys

import pandas as pd

sys.path.insert(0, os.getcwd())

# CTranslate2 busca libcublas.so.12 y el cargador de Colab no la encuentra: se precarga desde el paquete de pip.
spec = importlib.util.find_spec("nvidia")
for base in (spec.submodule_search_locations if spec else []):
    for ruta in Path(base).glob("cublas/lib/libcublas.so.12"):
        ctypes.CDLL(str(ruta), mode=ctypes.RTLD_GLOBAL)

from engine import load_config, task_dir
from engine.benchmark import field_accuracy, load_ground_truth, run_benchmark, run_ladder
from engine.core import process_audio
from engine.transcriber import cuda_available

config = load_config("config.yaml")
TIENE_GPU = cuda_available()
print("GPU disponible:", TIENE_GPU)
if not TIENE_GPU:
    print("⚠️ Sin GPU: el benchmark correrá solo en CPU. Actívala en Entorno de ejecución → Cambiar tipo de entorno.")

## 4. Clave de la API y audios

- **Clave (opcional):** solo se lee de los Secrets de Colab (`ANTHROPIC_API_KEY`) si `pipeline.extractor` no es `rules`. Nunca la escribas en una celda.
- **Audios:** `files.upload()` es opcional. Si subes archivos (`entrevista_01.m4a`, `Llamada 1.m4a`...) se copian a
  `tasks/*/audio/` con `scripts/organize_audio.py`; si no subes nada se usan los que ya trae el repo.

In [ ]:
# La clave de Anthropic solo hace falta para el extractor LLM (de pago). Con pipeline.extractor: rules (config.yaml)
# el notebook no usa la API y no hay que configurar nada.
if config["pipeline"]["extractor"] == "rules":
    print("Extractor por reglas (config.yaml): no se usa la API de Anthropic, no hace falta clave.")
else:
    try:
        from google.colab import userdata

        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        print("ANTHROPIC_API_KEY cargada desde los Secrets de Colab.")
    except Exception as e:
        print(f"Sin ANTHROPIC_API_KEY ({type(e).__name__}): se usará solo el extractor por reglas.")

In [ ]:
# Subida opcional de audios. files.upload() deja los archivos en la carpeta actual, así que se hace desde una carpeta temporal.
from google.colab import files

TEMPORAL = Path("/tmp/audios_subidos")
TEMPORAL.mkdir(parents=True, exist_ok=True)
PROYECTO = os.getcwd()
os.chdir(TEMPORAL)
subidos = files.upload()  # si no quieres subir nada, cancela o deja vacío: se usan los audios del repo
os.chdir(PROYECTO)

if subidos:
    !python scripts/organize_audio.py {TEMPORAL}
for t in ("task1_interview", "task2_sales"):
    print(t, "->", len(list(task_dir(config, t, "audio").glob("*"))), "audios")

## 5. Parámetros del benchmark

Por defecto corre **todo**: los modelos y precisiones de `config.yaml`, en GPU y CPU, con y sin glosario.
Para una prueba rápida, pon `LIMITE_AUDIOS = 2`. Las combinaciones no soportadas (por ejemplo `float16` en CPU) se omiten y
quedan anotadas en `benchmark_skipped.csv`.

In [ ]:
MODELOS = config["asr"]["models"]            # ["medium", "large-v3"]
PRECISIONES = config["asr"]["precisions"]    # ["float16", "int8"]
DISPOSITIVOS = ["cuda", "cpu"] if TIENE_GPU else ["cpu"]
GLOSARIO = ["off", "on"]                     # sin y con el glosario de dominio (initial_prompt)
LIMITE_AUDIOS = None                         # None = todos los audios; p. ej. 2 para una prueba rápida
MODELOS_CPU = None                           # None = los mismos modelos en CPU. La CPU de Colab es lenta (medium ~1,5 min por audio,
                                             # large-v3 bastante más): p. ej. ["medium"] para limitarla.

print("Combinaciones:", len(MODELOS) * len(PRECISIONES) * len(DISPOSITIVOS), "modelo×precisión×dispositivo, × glosario", GLOSARIO)

## 6. Benchmark (GPU + CPU)

Transcribe cada audio con cada combinación y mide WER/CER contra el guion (textos normalizados: minúsculas, sin
puntuación, sin espacios dobles), tiempo y VRAM. Guarda cada transcripción como JSON en `transcriptions_cache/` y los CSV
de métricas en `metrics/`.

In [ ]:
resultados = run_benchmark(config, models=MODELOS, precisions=PRECISIONES, devices=DISPOSITIVOS,
                           glossary=GLOSARIO, limit=LIMITE_AUDIOS, models_cpu=MODELOS_CPU)

## 7. Resultados del benchmark

- **Resumen:** WER/CER de corpus con intervalo de confianza del 95 % (bootstrap sobre los clips), tiempo y VRAM.
  `wer_numnorm` pasa antes los números hablados a dígitos ("cuatro mil quinientos" → 4500).
- **Efecto del glosario:** `delta_wer` negativo = el glosario ayuda; `ic_excluye_cero` dice si el efecto es distinguible de cero.
- **Cambios de texto:** ¿cambia la transcripción entre `float16` e `int8`, o entre GPU y CPU?

In [ ]:
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

cols = ["task", "model", "compute_type", "device", "glossary", "n_clips", "wer", "wer_ci_low", "wer_ci_high",
        "cer", "wer_numnorm", "elapsed_s", "vram_device_mb", "vram_mb"]
display(resultados["summary"][cols].round(3))

In [ ]:
print("Efecto del glosario (con − sin):")
display(resultados["glossary_effect"].round(3))
print("Combinaciones omitidas:")
display(resultados["skipped"])

In [ ]:
cambios = resultados["precision_changes"]
print("¿Cambió el texto? (por eje de comparación)")
if len(cambios):
    display(cambios.groupby(["task", "model", "eje", "config_a", "config_b"])["texto_cambio"].agg(["sum", "count"])
            .rename(columns={"sum": "clips_con_cambio", "count": "clips"}))
else:
    display("No hay pares de configuraciones comparables.")

## 8. Extracción con ambos caminos

Se toma, por tarea, la configuración con menor `wer_numnorm` y se extraen los campos de **sus** transcripciones
(guardadas en `transcriptions_cache/`) con el extractor por reglas (y con el LLM solo si `pipeline.extractor` no es `rules` y hay clave). Cada resultado se compara con
el ground truth y se verifica contra la transcripción.

In [ ]:
ESQUEMAS = {"task1_interview": "interview", "task2_sales": "sales"}
# Con pipeline.extractor: rules solo se usan las reglas (sin API); en otro caso, el LLM se suma si hay clave.
USAR_LLM = config["pipeline"]["extractor"] != "rules" and bool(os.environ.get("ANTHROPIC_API_KEY"))
EXTRACTORES = ["rules"] + (["llm"] if USAR_LLM else [])
print("Extractores:", EXTRACTORES)


def mejor_configuracion(task):
    s = resultados["summary"]
    return s[s["task"] == task].sort_values("wer_numnorm").iloc[0]


def cargar_transcripciones(task, fila):
    sufijo = "gloss" if fila["glossary"] == "on" else "nogloss"
    patron = f"*__{fila['model']}_{fila['compute_type']}_{fila['device']}_{sufijo}.json"
    datos = [json.loads(p.read_text(encoding="utf-8")) for p in sorted(task_dir(config, task, "transcriptions_cache").glob(patron))]
    return {d["clip"]: d["transcript"] for d in datos}

In [ ]:
filas_extraccion, filas_validacion, procesados = [], [], {}
for task, schema in ESQUEMAS.items():
    mejor = mejor_configuracion(task)
    textos = cargar_transcripciones(task, mejor)
    gt = load_ground_truth(config, task)
    print(f"{task}: {mejor['model']} {mejor['compute_type']} {mejor['device']} glosario={mejor['glossary']} (WER {mejor['wer_numnorm']:.1%}), {len(textos)} clips")
    for clip, texto in textos.items():
        for ext in EXTRACTORES:
            r = process_audio(None, schema, extractor=ext, transcript=texto)  # sin ASR: usa la transcripción de la caché
            procesados[(task, clip, ext)] = r
            if r["error"]:
                print(f"  ⚠️ {clip} [{ext}]: {r['error']}")
                continue
            acc = field_accuracy(config, task, clip, r["fields"], gt)
            for campo, valor in r["fields"].items():
                filas_extraccion.append({"task": task, "clip": clip, "extractor": ext, "campo": campo,
                                         "valor": json.dumps(valor, ensure_ascii=False),
                                         "correcto": acc.get(campo), "verificacion": r["verified_fields"][campo]})
            fila_val = {"task": task, "clip": clip, "extractor": ext,
                        "campos_sin_verificar": ", ".join(c for c, v in r["verified_fields"].items() if v == "sin_verificar")}
            if schema == "sales":
                a = r["diagnostics"]["arithmetic"]
                gt_incorrecto = gt[clip]["total_es_incorrecto"] == "Sí"
                fila_val.update(total_esperado=a["expected_total"], total_dicho=a["total_dicho"], aritmetica=a["status"],
                                diagnostico=a["diagnosis"], gt_total_incorrecto=gt_incorrecto,
                                detecta_bien=(a["status"] == "discrepancia") == gt_incorrecto if a["status"] != "incompleto" else None)
            filas_validacion.append(fila_val)

extraccion, validacion = pd.DataFrame(filas_extraccion), pd.DataFrame(filas_validacion)
print("\nExactitud por campo (fracción de clips correctos):")
display(extraccion.pivot_table(index=["task", "campo"], columns="extractor", values="correcto", aggfunc="mean").round(2))

## 9. Validación

- **Verificación literal:** un campo queda `sin_verificar` si su valor no está en la transcripción. Con el LLM suele pasar
  cuando Whisper deforma un término ("Payton") y el modelo lo corrige a "Python": el valor es correcto pero no es literal.
- **Aritmética (ventas):** compara el total dicho con cantidad × precio − promoción. Si no coincide, el diagnóstico
  distingue `hablante` (las cifras aparecen en la transcripción, pero el total dicho está mal) de `asr` (alguna cifra no aparece).

In [ ]:
print("Campos sin verificar literalmente:")
display(validacion[validacion["campos_sin_verificar"] != ""][["task", "clip", "extractor", "campos_sin_verificar"]])

In [ ]:
if "aritmetica" in validacion:
    print("Aritmética de ventas:")
    display(validacion[validacion["task"] == "task2_sales"][["clip", "extractor", "total_esperado", "total_dicho", "aritmetica",
                                                           "diagnostico", "gt_total_incorrecto", "detecta_bien"]])

## 10. Escalera de degradación (ventas)

Cada llamada se degrada en 11 niveles (más el `original`) con `tasks/task2_sales/degradation/build_ladder.py`: ruido blanco con
SNR 20/10/5/0 dB, banda telefónica, MP3 a 32/16/8 kbps y volumen -20/-30/-40 dB. Se transcribe cada nivel y se mide **cuánto sube el
WER respecto al original** (`ladder_curve.csv`, con IC 95 % por bootstrap sobre los clips).

- Si ya tienes los `.wav` (por ejemplo los generaste en local), súbelos a `tasks/task2_sales/degradation/<nivel>/` junto con
  `ladder_manifest.csv` y la celda de generación los respeta. Si no, se generan aquí: es reproducible (semilla 0).
- Por defecto: GPU, `float16`, sin glosario y los dos modelos. Cambia los parámetros de la segunda celda para otra cosa.

In [ ]:
# Generación de la escalera (solo si todavía no hay .wav en tasks/task2_sales/degradation/).
RUN_ESCALERA = True
LADDER_DIR = task_dir(config, "task2_sales", "degradation")
AUDIO_VENTAS = task_dir(config, "task2_sales", "audio")
if RUN_ESCALERA and not any(LADDER_DIR.glob("*/*.wav")):
    !python tasks/task2_sales/degradation/build_ladder.py --input_dir "{AUDIO_VENTAS}"

In [ ]:
LADDER_MODELOS = ["medium", "large-v3"]
LADDER_PRECISIONES = ["float16"]
LADDER_DISPOSITIVOS = ["cuda"] if TIENE_GPU else ["cpu"]   # en CPU es muy lento
LADDER_GLOSARIO = ["off"]
LADDER_NIVELES = None     # None = todos; p. ej. ["original", "ruido_snr20", "ruido_snr10", "ruido_snr5"]
LADDER_LIMITE = None      # None = todos los clips de cada nivel

if RUN_ESCALERA:
    escalera = run_ladder(config, LADDER_DIR, "task2_sales", levels=LADDER_NIVELES, models=LADDER_MODELOS,
                          precisions=LADDER_PRECISIONES, devices=LADDER_DISPOSITIVOS, glossary=LADDER_GLOSARIO,
                          limit=LADDER_LIMITE)
    # El manifiesto documenta exactamente qué se transcribió (SNR logrado, RMS, pico): viaja junto con las métricas.
    shutil.copy(LADDER_DIR / "ladder_manifest.csv", task_dir(config, "task2_sales", "metrics") / "ladder_manifest.csv")

In [ ]:
if RUN_ESCALERA:
    print("WER de corpus por nivel (menor = mejor):")
    display(escalera["summary"].pivot_table(index=["family", "severity", "level"], columns=["model", "compute_type"],
                                            values="wer").round(3))
    print("Cuánto sube el WER respecto al original (delta_wer, con IC 95 %):")
    display(escalera["curve"][["model", "level", "severity", "wer_original", "wer_level", "delta_wer", "delta_ci_low",
                               "delta_ci_high", "ic_excluye_cero"]].round(3))

## 11. Guardar y descargar todo

Se guardan `extraction_results.csv` y `validation_results.csv` junto a los CSV del benchmark en `tasks/*/metrics/`
(las transcripciones ya están en `transcriptions_cache/`) y se descarga un `.zip` con todo.

In [ ]:
for task in ESQUEMAS:
    mdir = task_dir(config, task, "metrics")
    extraccion[extraccion["task"] == task].to_csv(mdir / "extraction_results.csv", index=False, encoding="utf-8")
    validacion[validacion["task"] == task].to_csv(mdir / "validation_results.csv", index=False, encoding="utf-8")
    print(task, "->", sorted(p.name for p in mdir.glob("*.csv")))
    print("   transcripciones:", len(list(task_dir(config, task, "transcriptions_cache").glob("*.json"))), "JSON")

In [ ]:
# Empaqueta metrics/ y transcriptions_cache/ de ambas tareas en un .zip y lo descarga.
from google.colab import files

ZIP = Path("/tmp/resultados_tarea4")
RAIZ = task_dir(config, "task1_interview", "metrics").parent.parent  # .../tasks
temporal = Path("/tmp/paquete_resultados")
shutil.rmtree(temporal, ignore_errors=True)
for task in ESQUEMAS:
    for sub in ("metrics", "transcriptions_cache"):
        shutil.copytree(RAIZ / task / sub, temporal / task / sub, ignore=shutil.ignore_patterns(".gitkeep"))
archivo = shutil.make_archive(str(ZIP), "zip", root_dir=temporal)
print("Paquete:", archivo)
files.download(archivo)